In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import h5py
import ipynbname

import pandas as pd

import ferdinand.utils as utl   
import ferdinand.sqlite_utils as sqlutl
import ferdinand.model_setup as modsetup
import ferdinand.preprocess_and_qc as ppqc

from pathlib import Path
from types import SimpleNamespace

from tensorflow.keras.optimizers import Adam

# Purpose and Documentation

This notebook provides a standardized structure for project notebooks, including path setup, logging, and metadata tracking. </br>
Use it as a starting point for new notebooks to ensure consistency, reproducibility, and clarity across the project.

# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['data_directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'])
os.makedirs(dest_dir, exist_ok=True)
dest_path = os.path.join(dest_dir, dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_name = ipynbname.name()  
notebook_idx = notebook_name.split('_')[0]

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_name}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, f"{config['sqlite_db']}.db")

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlutl.connect_db(sqlite_file)
    if conn is not None:
        print(f"connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

## Load model

In [ ]:
# check if the expected AI model file exists and can be read as HDF5
try: 
    model_dir = os.path.join(config['project_root'], config['models_dir'], config['run_name'])
    os.makedirs(model_dir, exist_ok=True)
    model_file = os.path.join(model_dir, config['angle_detection_model'])
    if not os.path.exists(model_file): 
        print(f"ERROR: no angle detection model {model_file} found!")
    else:
        print(f" model size: {os.path.getsize(model_file)}")
        try:
            with h5py.File(model_file, "r") as f: 
                print("keys in HDF5 file:", list(f.keys()))
        except Exception as e:
            print("ERROR: Not HDF5:\n", e)

except NameError as e:
    print(f"model path could not be set up: {e}")

In [ ]:
# load the model architecture and weights
model = modsetup.create_model()
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='categorical_crossentropy', 
              metrics=[modsetup.angle_error])
if os.path.exists(model_file):
    model.load_weights(model_file)

# First angle prediction

In [ ]:
# get list of curated images to predict orientation angle for
curated_images = utl.select_jpegs_by_columns(conn=conn, 
                                             db_table='impc_data', 
                                             filters={'keep': 'yes', 'overexposed': 'no', 'overall_overexposed': 'no'}, 
                                             source_path=dirs.preprocessed_dir) 
for _img in curated_images:
    angle = ppqc.predict_orientation_angle(conn=conn, 
                                           db_table='impc_data', 
                                           model=model, 
                                           image_file_path=_img,
                                           db_column_name='fst_angle_prediction')
    print(f" first angle prediction for image {_img}: {angle}")

# Rotate images

In [ ]:
# add column to database to flag if image has been rotated
sqlutl.add_column_to_table(conn=conn, 
                           db_table='impc_data', 
                           column_name='rotated', 
                           column_type='TEXT',
                           default_value='no', 
                           logger=logger)

# get list of curated images to rotate
imgs_to_rotate = utl.select_jpegs_by_columns(conn=conn, 
                                             db_table='impc_data',
                                             filters={'keep': 'yes', 
                                                      'overexposed': 'no',
                                                      'overall_overexposed': 'no', 
                                                      'rotated': 'no'}, 
                                             source_path=dirs.preprocessed_dir)

if len(imgs_to_rotate) > 0:
    print(f" number of curated images with no exposure to rotate: {len(imgs_to_rotate)}")

    # rotate images and update database
    ppqc.load_and_rotate_images(conn, 
                                db_table='impc_data',
                                image_files=imgs_to_rotate, 
                                source_path=dirs.preprocessed_dir,
                                target_path=dirs.rotated_dir,
                                logger=logger)
else:
    print(" no images to rotate")

In [ ]:
# check that all rotated files are present
files = utl.select_jpegs_by_column(conn, 
                                   db_table='impc_data', 
                                   column='rotated', 
                                   value='yes', 
                                   source_path=dirs.rotated_dir)
for file in files:
    if not os.path.exists(file):
        print(f" missing file {file}")

# Second angle prediction

For evaluation purposes. </br> 
**Rotation QC**: How many images failed alignment, and what was their angular deviation from target (>10°)?

In [ ]:
# add column to database to flag if image has been rotated
sqlutl.add_column_to_table(conn=conn, 
                           db_table='impc_data', 
                           column_name='snd_angle_prediction', 
                           column_type='INTEGER', 
                           logger=logger)

# after rotation, we can check the second angle prediction for the rotated images
rotated_images = utl.select_jpegs_by_columns(conn=conn, 
                                             db_table='impc_data', 
                                             filters={'rotated': 'yes', 'snd_angle_prediction': None}, 
                                             source_path=dirs.rotated_dir) 

for _img in rotated_images:
    angle = ppqc.predict_orientation_angle(conn=conn, 
                                           db_table='impc_data',
                                           model=model, 
                                           image_file_path=_img, 
                                           db_column_name='snd_angle_prediction')
    print(f" second angle prediction for image {_img}: {angle}")

In [ ]:
# get all rows where 'rotated' is 'yes'
rotated_df = sqlutl.select_rows_by_column(conn=conn, 
                                          db_table='impc_data', 
                                          column='rotated', 
                                          value='yes')

In [ ]:
# function to calculate and print the percentage of non-aligned images per center
def nonaligned_percentage(_center: str, _data_dir: str, _data: pd.DataFrame, _angle_col: str) -> None:
    
    total = len(rotated_df[rotated_df.center == _center])

    _df = _data[_data.center == _center]
    nonaligned = len(_df)

    print(f" number of non-aligned images in center {_center}: {nonaligned} from {total} ({int((nonaligned/total)*100) if total > 0 else 0}%)")
    print(f" orientations of non-aligned images: {_df[_angle_col].unique()}\n")

    if len(_df) > 0:
        rnd_nonaligned = _df.sample(n=int(0.01*nonaligned))

        rnd_nonaligned_images = []
        rnd_nonaligned_pred_angles = []
        for idx, row in rnd_nonaligned.iterrows():
            rnd_nonaligned_images.append(os.path.join(utl.build_local_image_dir(_data_dir, 
                                                                               row['center'], 
                                                                               row['cohort_type'], 
                                                                               row['gene_symbol'], 
                                                                               row['sex']), f"{row['omero_id']}.jpg"))
            rnd_nonaligned_pred_angles.append(row[_angle_col])

        if len(rnd_nonaligned_images) > 0: 
            utl.plot_image_grid(image_files=rnd_nonaligned_images, 
                               cols=5, 
                               titles=rnd_nonaligned_pred_angles)

In [ ]:
# create dataframe of non-aligned images (angle prediction between 10 and 350 degrees)
nonaligned_df = rotated_df[(rotated_df['snd_angle_prediction'] >= 10) & (rotated_df['snd_angle_prediction'] <= 350)]
nonaligned_df.head(5)

In [ ]:
# summarize the percentage of non-aligned images per center
for center in config['centers']:
    nonaligned_percentage(_center=center, 
                          _data_dir=dirs.rotated_dir, 
                          _data=nonaligned_df, 
                          _angle_col='snd_angle_prediction')

# Create thumbnail gallery

In [ ]:
# create an HTML gallery directory
gallery_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'thumbnails_gallery')
os.makedirs(gallery_dir, exist_ok=True)

## Rotated images

In [ ]:
# select all rotated images
rows = sqlutl.select_rows_by_column(conn, 'impc_data', 'rotated', 'yes')

In [ ]:
# create thumbnails for all rotated images
utl.create_and_save_thumbnails(
    data=rows,
    centers=config['centers'],
    path2images=dirs.rotated_dir,
    path2thumbnails=dirs.rotated_thumbnails_dir
)

In [ ]:
# create HTML gallery of all rotated images
utl.create_thumbnail_gallery(
    data=rows.sort_values(by='snd_angle_prediction', ascending=True),
    html_file_name=f'{notebook_idx}_all_rotated_images.html',
    title='Overview of Rotated Images',
    html_dir=gallery_dir,
    thumbnails_dir=dirs.rotated_thumbnails_dir,
    source_dir=dirs.rotated_dir, 
    alignment_details=True)

In [ ]:
# create center-specific galleries of rotated images
for center in config['centers']:
    center_df = rows[rows['center'] == center].sort_values(by='snd_angle_prediction', ascending=True)
    if not center_df.empty:
        center_stripped = center.replace(" ", "_")
        center_gallery_dir = os.path.join(gallery_dir, center_stripped)
        os.makedirs(center_gallery_dir, exist_ok=True)
        utl.create_thumbnail_gallery(
            data=center_df,
            html_file_name=f'{notebook_idx}_all_rotated_images_{center_stripped}.html',
            title=f'Overview of Rotated Images from {center}',
            html_dir=center_gallery_dir,
            thumbnails_dir=dirs.rotated_thumbnails_dir,
            source_dir=dirs.rotated_dir, 
            alignment_details=True)

## Non-aligned images (rotation QC failed)

In [ ]:
# create HTML gallery of non-aligned images
utl.create_thumbnail_gallery(
    data=nonaligned_df.sort_values(by='snd_angle_prediction', ascending=True),
    html_file_name=f'{notebook_idx}_non_aligned_images.html',
    title='Overview of Non-Aligned Images',
    html_dir=gallery_dir,
    thumbnails_dir=dirs.rotated_thumbnails_dir,
    source_dir=dirs.rotated_dir)

In [ ]:
# create center-specific galleries of non-aligned images
for center in config['centers']:
    center_df = nonaligned_df[nonaligned_df['center'] == center].sort_values(by='snd_angle_prediction', ascending=True)
    if not center_df.empty:
        center_stripped = center.replace(" ", "_")
        center_gallery_dir = os.path.join(gallery_dir, center_stripped)
        os.makedirs(center_gallery_dir, exist_ok=True)
        utl.create_thumbnail_gallery(
            data=center_df,
            html_file_name=f'{notebook_idx}_non_aligned_images_{center_stripped}.html',
            title=f'Overview of Non-Aligned Images from {center}',
            html_dir=center_gallery_dir,
            thumbnails_dir=dirs.rotated_thumbnails_dir,
            source_dir=dirs.rotated_dir)